# 动态规划算法学习笔记

本文档基于《动手学强化学习》第 4 章“动态规划算法”整理，原文链接：<https://hrl.boyuai.com/chapter/1/%E5%8A%A8%E6%80%81%E8%A7%84%E5%88%92%E7%AE%97%E6%B3%95/>。

我把这篇笔记分成四部分：

1. 先用自己的话总结动态规划在强化学习中的使用前提。
2. 再整理策略迭代和价值迭代的核心公式。
3. 接着用 Cliff Walking 这个小环境把代码跑通。
4. 最后用 FrozenLake 对照理解“随机转移”对策略的影响。

## 1. 我对动态规划的理解

动态规划（Dynamic Programming, DP）在强化学习里并不是“直接和环境交互学出来”的，而是建立在**已知环境模型**的前提上：

- 已知状态转移概率 $P(s' \mid s, a)$
- 已知奖励函数 $R(s, a, s')$
- 状态空间和动作空间通常都是有限的

这意味着 DP 更像是一个“白盒求解器”：一旦 MDP 的结构都已知，我们就可以反复利用贝尔曼方程更新每个状态的价值，直到收敛。

所以我这里把它理解成：

- **策略迭代**：先把当前策略的价值算清楚，再按价值改进策略。
- **价值迭代**：不把某个策略评估到特别精确，直接朝着最优价值方向更新。

它们的共同点是都在“全状态空间”上做更新；不同点是是否显式地维护并完整评估一个策略。

## 2. 关键公式整理

### 2.1 策略评估

给定策略 $\pi$，状态价值函数满足贝尔曼期望方程：

$$
v_{\pi}(s)=\sum_a \pi(a\mid s) \sum_{s',r} p(s',r\mid s,a) \left[r + \gamma v_{\pi}(s')\right]
$$

这一步的直觉是：

- 当前状态值 = 当前策略下所有动作的期望
- 每个动作的价值又依赖下一个状态的价值
- 因此可以通过反复迭代达到不动点

### 2.2 策略提升

如果已经得到 $v_{\pi}(s)$，就可以计算动作价值：

$$
q_{\pi}(s,a)=\sum_{s',r} p(s',r\mid s,a)\left[r + \gamma v_{\pi}(s')\right]
$$

然后做贪心改进：

$$
\pi'(s)=\arg\max_a q_{\pi}(s,a)
$$

如果新旧策略一致，说明已经稳定，也就到达了最优策略。

### 2.3 价值迭代

价值迭代直接使用贝尔曼最优方程：

$$
v_*(s)=\max_a \sum_{s',r} p(s',r\mid s,a)\left[r + \gamma v_*(s')\right]
$$

与策略迭代相比，它把“评估 + 提升”压缩成了一步：每次更新都直接取最大动作价值。我的理解是，价值迭代更像在直接逼近最优值函数，而策略迭代更像在一轮轮“改策略”。

In [ ]:
from pprint import pprint

## 3. 悬崖漫步环境：先把环境和打印函数准备好

原文先在一个 4x12 的网格世界里演示 DP。这个环境的特点很清楚：

- 起点在左下角，终点在右下角
- 底边中间一排是悬崖
- 每走一步奖励都是 -1
- 掉进悬崖奖励是 -100，并立即结束

我觉得这个环境很适合学习动态规划，因为：

- 状态空间很小，便于完全枚举
- 奖励结构很明确，价值函数有直观解释
- 最优策略既体现“尽快到终点”，又体现“避开高风险区域”

In [ ]:
class CliffWalkingEnv:
    """悬崖漫步环境。状态编号按行优先展开。"""

    def __init__(self, ncol=12, nrow=4):
        self.ncol = ncol
        self.nrow = nrow
        self.P = self._create_transition_matrix()

    def _create_transition_matrix(self):
        P = [[[] for _ in range(4)] for _ in range(self.nrow * self.ncol)]
        change = [(0, -1), (0, 1), (-1, 0), (1, 0)]  # 上、下、左、右

        for i in range(self.nrow):
            for j in range(self.ncol):
                state = i * self.ncol + j
                for action, (dx, dy) in enumerate(change):
                    if i == self.nrow - 1 and j > 0:
                        P[state][action] = [(1.0, state, 0, True)]
                        continue

                    next_x = min(self.ncol - 1, max(0, j + dx))
                    next_y = min(self.nrow - 1, max(0, i + dy))
                    next_state = next_y * self.ncol + next_x
                    reward = -1
                    done = False

                    if next_y == self.nrow - 1 and next_x > 0:
                        done = True
                        if next_x != self.ncol - 1:
                            reward = -100

                    P[state][action] = [(1.0, next_state, reward, done)]
        return P


def print_agent(agent, action_meaning, disaster=None, end=None):
    disaster = set(disaster or [])
    end = set(end or [])

    print("状态价值：")
    for i in range(agent.env.nrow):
        row = []
        for j in range(agent.env.ncol):
            value = agent.v[i * agent.env.ncol + j]
            row.append(f"{value:6.3f}")
        print(" ".join(row))

    print("\n策略：")
    for i in range(agent.env.nrow):
        row = []
        for j in range(agent.env.ncol):
            state = i * agent.env.ncol + j
            if state in disaster:
                row.append("****")
            elif state in end:
                row.append("EEEE")
            else:
                probs = agent.pi[state]
                row.append("".join(action_meaning[k] if probs[k] > 0 else "o" for k in range(len(action_meaning))))
        print(" ".join(row))

## 4. 策略迭代：评估当前策略，再做贪心提升

我把策略迭代拆成三个动作：

1. `policy_evaluation`：把当前策略对应的状态价值函数迭代出来。
2. `policy_improvement`：对每个状态计算动作价值并取贪心动作。
3. `policy_iteration`：不停交替执行上面两步，直到策略稳定。

实现时我保留了一个我认为很重要的细节：**如果多个动作并列最优，就让它们均分概率**。这样既和原文一致，也能更真实地展示“最优策略未必唯一”。

In [ ]:
class PolicyIteration:
    def __init__(self, env, theta=1e-3, gamma=0.9):
        self.env = env
        self.theta = theta
        self.gamma = gamma
        self.v = [0.0] * (env.ncol * env.nrow)
        self.pi = [[0.25, 0.25, 0.25, 0.25] for _ in range(env.ncol * env.nrow)]

    def _calc_qsa(self, state, action):
        qsa = 0.0
        for p, next_state, reward, done in self.env.P[state][action]:
            qsa += p * (reward + self.gamma * self.v[next_state] * (1 - done))
        return qsa

    def policy_evaluation(self):
        evaluation_rounds = 0
        while True:
            max_diff = 0.0
            new_v = [0.0] * (self.env.ncol * self.env.nrow)

            for s in range(self.env.ncol * self.env.nrow):
                state_value = 0.0
                for a in range(4):
                    state_value += self.pi[s][a] * self._calc_qsa(s, a)
                new_v[s] = state_value
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))

            self.v = new_v
            evaluation_rounds += 1
            if max_diff < self.theta:
                break
        return evaluation_rounds

    def policy_improvement(self):
        policy_stable = True
        for s in range(self.env.nrow * self.env.ncol):
            old_action_distribution = self.pi[s][:]
            qsa_list = [self._calc_qsa(s, a) for a in range(4)]
            max_q = max(qsa_list)
            max_count = qsa_list.count(max_q)
            self.pi[s] = [1 / max_count if q == max_q else 0 for q in qsa_list]
            if self.pi[s] != old_action_distribution:
                policy_stable = False
        return policy_stable

    def policy_iteration(self, verbose=True):
        history = []
        while True:
            rounds = self.policy_evaluation()
            stable = self.policy_improvement()
            history.append(rounds)
            if verbose:
                print(f"第 {len(history)} 次策略评估共迭代 {rounds} 轮")
            if stable:
                break
        return history

In [ ]:
cliff_env = CliffWalkingEnv()
action_meaning = ["^", "v", "<", ">"]

pi_agent = PolicyIteration(cliff_env, theta=1e-3, gamma=0.9)
pi_history = pi_agent.policy_iteration()

print("每次策略评估的迭代轮数:", pi_history)
print_agent(pi_agent, action_meaning, disaster=range(37, 47), end=[47])

### 策略迭代实验后的记录

跑完上面的代码后，我最关注的是两件事：

- 策略评估的轮数通常比较多，这说明“把一个策略评估到接近收敛”本身就有不小计算量。
- 最终策略会沿着悬崖上方一行向右走，并不是紧贴悬崖冲过去，因为那样虽然步数短，但风险太大。

这也让我更容易理解价值函数的含义：它不是单纯的“还要走多少步”，而是把**未来所有奖励与风险折扣之后的综合预期**。

## 5. 价值迭代：直接逼近最优价值函数

价值迭代和策略迭代最像的一点是都在算 $Q(s,a)$；最大的不同是：

- 策略迭代对 $Q(s,a)$ 做的是“按当前策略加权求和”
- 价值迭代对 $Q(s,a)$ 做的是“直接取最大值”

所以我会把它记成一句话：

> 策略迭代先学会评价一个策略，价值迭代直接朝着最优策略走。

In [ ]:
class ValueIteration:
    def __init__(self, env, theta=1e-3, gamma=0.9):
        self.env = env
        self.theta = theta
        self.gamma = gamma
        self.v = [0.0] * (env.ncol * env.nrow)
        self.pi = [None for _ in range(env.ncol * env.nrow)]

    def _calc_qsa(self, state, action):
        qsa = 0.0
        for p, next_state, reward, done in self.env.P[state][action]:
            qsa += p * (reward + self.gamma * self.v[next_state] * (1 - done))
        return qsa

    def value_iteration(self, verbose=True):
        iteration = 0
        while True:
            max_diff = 0.0
            new_v = [0.0] * (self.env.ncol * self.env.nrow)

            for s in range(self.env.ncol * self.env.nrow):
                qsa_list = [self._calc_qsa(s, a) for a in range(4)]
                new_v[s] = max(qsa_list)
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))

            self.v = new_v
            iteration += 1
            if max_diff < self.theta:
                break

        if verbose:
            print(f"价值迭代共进行了 {iteration} 轮")
        self.get_policy()
        return iteration

    def get_policy(self):
        for s in range(self.env.nrow * self.env.ncol):
            qsa_list = [self._calc_qsa(s, a) for a in range(4)]
            max_q = max(qsa_list)
            max_count = qsa_list.count(max_q)
            self.pi[s] = [1 / max_count if q == max_q else 0 for q in qsa_list]

In [ ]:
vi_agent = ValueIteration(cliff_env, theta=1e-3, gamma=0.9)
vi_rounds = vi_agent.value_iteration()

print_agent(vi_agent, action_meaning, disaster=range(37, 47), end=[47])
print("\n两种方法的价值函数是否一致:", all(abs(a - b) < 1e-6 for a, b in zip(pi_agent.v, vi_agent.v)))
print("两种方法的策略是否一致:", pi_agent.pi == vi_agent.pi)

### 我对两种算法差异的总结

| 对比点 | 策略迭代 | 价值迭代 |
| --- | --- | --- |
| 核心思想 | 完整评估当前策略，再改进 | 每轮直接取最优动作 |
| 是否显式维护策略 | 是 | 最后再恢复 |
| 单次迭代代价 | 通常更大 | 通常更小 |
| 直觉 | “先看清楚，再行动” | “边估边选最优” |

在 Cliff Walking 这种小环境里，两者最后会给出相同的最优价值和最优策略；但如果只看迭代效率，价值迭代通常会更直接。

## 6. FrozenLake：随机转移会改变最优策略的直觉

原文还用 FrozenLake 做了一个很好的补充。这个环境和悬崖漫步的最大区别是：**动作不是确定性的**。智能体在冰面上可能会滑到别的位置，因此最优动作未必是“几何上最短”的动作。

例如在接近终点的位置，看起来“向右”最合理，但如果向右后有较大概率滑进危险区域，那么“向下”这种更保守的动作反而可能更优。

下面这段代码会优先尝试 `gymnasium`，再尝试 `gym`。如果本地没装相关包，它会自动跳过。原文使用的是 `FrozenLake-v0`，但在较新的环境里通常需要 `FrozenLake-v1`。

In [ ]:
def make_frozen_lake_env():
    gym_module = None
    env = None
    env_name = None

    try:
        import gymnasium as gym_module
        for candidate in ("FrozenLake-v1", "FrozenLake-v0"):
            try:
                env = gym_module.make(candidate)
                env_name = candidate
                break
            except Exception:
                pass
    except ImportError:
        try:
            import gym as gym_module
            for candidate in ("FrozenLake-v1", "FrozenLake-v0"):
                try:
                    env = gym_module.make(candidate)
                    env_name = candidate
                    break
                except Exception:
                    pass
        except ImportError:
            return None, None

    if env is None:
        return None, None

    env = env.unwrapped
    if hasattr(env, "desc"):
        env.nrow, env.ncol = env.desc.shape
    return env, env_name


frozen_env, frozen_env_name = make_frozen_lake_env()
if frozen_env is None:
    print("当前环境没有安装 gym/gymnasium，FrozenLake 示例先保留代码，不执行。")
else:
    holes = set()
    ends = set()
    for s in frozen_env.P:
        for a in frozen_env.P[s]:
            for p, next_state, reward, done in frozen_env.P[s][a]:
                if reward == 1.0:
                    ends.add(next_state)
                if done:
                    holes.add(next_state)
    holes -= ends

    print("FrozenLake 环境版本:", frozen_env_name)
    print("冰洞状态:", holes)
    print("目标状态:", ends)
    print("状态 14 的转移信息:")
    pprint(frozen_env.P[14])

In [ ]:
if frozen_env is not None:
    frozen_action_meaning = ["<", "v", ">", "^"]

    frozen_pi_agent = PolicyIteration(frozen_env, theta=1e-5, gamma=0.9)
    frozen_pi_agent.policy_iteration(verbose=False)
    print("策略迭代结果：")
    print_agent(frozen_pi_agent, frozen_action_meaning, disaster=holes, end=ends)

    frozen_vi_agent = ValueIteration(frozen_env, theta=1e-5, gamma=0.9)
    frozen_vi_agent.value_iteration(verbose=False)
    print("\n价值迭代结果：")
    print_agent(frozen_vi_agent, frozen_action_meaning, disaster=holes, end=ends)

## 7. 最后的总结

整理完这一章后，我对 DP 的理解更清楚了：

- 它不是“靠采样学”的方法，而是“已知模型后直接算”的方法。
- 策略迭代和价值迭代的本质都来自贝尔曼方程，只是更新方式不同。
- 在确定性环境里，最优策略往往更符合几何直觉；在随机环境里，最优策略会把“风险”显式考虑进去。
- DP 是理解强化学习的好起点，因为它把“价值函数”“策略改进”“最优性”这些概念都讲得非常具体。

如果后面继续学时序差分、Q-learning 或 DQN，我会把它们和这一章对照着看：那些方法本质上是在**不知道环境模型时，用采样近似地做这里的事情**。